# Bronze Layer Validation

Runs one consolidated validation after all AdventureWorks Auto Loader notebooks have completed.

This avoids repeating the same row-count and checkpoint validation logic inside every entity notebook.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F
from functools import reduce

In [0]:
results = []

for entity_name, cfg in ENTITY_CONFIG.items():
    target_table = cfg["target_table"]

    if not spark.catalog.tableExists(target_table):
        results.append(
            (entity_name, target_table, "MISSING", None, None, None, None)
        )
        continue

    summary = (
        spark.table(target_table)
        .agg(
            F.count("*").alias("row_count"),
            F.countDistinct("_source_file").alias("source_file_count"),
            F.min("_ingestion_timestamp").alias("first_ingestion"),
            F.max("_ingestion_timestamp").alias("last_ingestion"),
        )
        .first()
    )

    results.append(
        (
            entity_name,
            target_table,
            "OK",
            summary["row_count"],
            summary["source_file_count"],
            summary["first_ingestion"],
            summary["last_ingestion"],
        )
    )

validation_df = spark.createDataFrame(
    results,
    [
        "entity",
        "target_table",
        "status",
        "row_count",
        "source_file_count",
        "first_ingestion",
        "last_ingestion",
    ],
)

display(validation_df.orderBy("entity"))

## Duplicate business-key check

In [0]:
duplicate_results = []

for entity_name, cfg in ENTITY_CONFIG.items():
    target_table = cfg["target_table"]
    keys = cfg["business_keys"]

    if not spark.catalog.tableExists(target_table):
        continue

    duplicate_count = (
        spark.table(target_table)
        .groupBy(*keys)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    duplicate_results.append(
        (entity_name, ", ".join(keys), duplicate_count)
    )

display(
    spark.createDataFrame(
        duplicate_results,
        ["entity", "business_keys", "duplicate_key_groups"],
    ).orderBy("entity")
)

## Expected result

- Every configured entity should have `status = OK`.
- `row_count` should be greater than zero.
- `source_file_count` should be at least one.
- Duplicate business-key groups should be zero for the initial official AdventureWorks load.

Auto Loader checkpoints ensure that rerunning an entity notebook without a new source file does not append the same file again.